# 1. Objects, zones, and the one mutation primitive

`manalab.engine` is a Magic rules engine. This notebook covers its foundation:
what a thing *is* while a game is running, where things live, and the single
operation that moves them.

Everything else in the engine — combat, the stack, priority — is built on this,
so it is worth being slow here.

**The one idea to take away:** a card changing zones becomes a *new object* with
a new id, and the old one is deleted. That is a real Magic rule (CR 400.7), and
enforcing it mechanically is what makes a whole class of bug impossible.

In [1]:
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "data" / "cards.duckdb").exists() and REPO != REPO.parent:
    REPO = REPO.parent

from manalab.cards.mana import parse_cost
from manalab.cards.model import Card
from manalab.cards.store import CardStore

store = CardStore(REPO / "data" / "cards.duckdb")


def creature(name, power, toughness, *keywords, cost="{1}{G}"):
    # A synthetic creature, so a demo shows a rule and not a card.
    return Card(name.lower().replace(" ", "-"), name, parse_cost(cost),
                "Creature - Test", power=str(power), toughness=str(toughness),
                keywords=frozenset(keywords))


FOREST = store.by_name("Forest")

In [2]:
from manalab.engine.zones import Zone


def show(state, title=""):
    # Print the whole visible game state. The workhorse of these notebooks.
    if title:
        print(f"--- {title}")
    print(f"    turn {state.turn}, active P{state.active_player}")
    for player in state.players:
        flags = " LOST" if player.has_lost else ""
        print(f"    P{player.index}: {player.life} life{flags}"
              f"   hand={len(state.hand(player.index))}"
              f"  library={len(state.library(player.index))}"
              f"  graveyard={len(state.graveyard(player.index))}")
        for obj in state.battlefield(player.index):
            marks = []
            if obj.tapped:
                marks.append("tapped")
            if obj.damage:
                marks.append(f"dmg{obj.damage}")
            if obj.deathtouched:
                marks.append("deathtouched")
            if obj.summoning_sick:
                marks.append("sick")
            if obj.attacking is not None:
                marks.append(f"attacking P{obj.attacking}")
            if obj.blocking is not None:
                marks.append("blocking")
            if obj.counters:
                marks.extend(f"{k}x{v}" for k, v in sorted(obj.counters.items()))
            body = f"{obj.power}/{obj.toughness}" if obj.is_creature else "land"
            suffix = f"  [{' '.join(marks)}]" if marks else ""
            print(f"        #{obj.id} {obj.card.name:<22} {body:<6}{suffix}")
    stack = state.zone(Zone.STACK)
    if stack:
        print(f"    stack (top last): {[o.card.name for o in stack]}")

## Card vs GameObject

Two types, and the split matters for both correctness and speed.

- **`Card`** is frozen and *shared*. Every copy of Forest in every deck in every
  game is literally the same Python object. It holds what is printed: cost, type
  line, power, keywords.
- **`GameObject`** is mutable and *per-instance*. It holds what is true right
  now: which zone, who controls it, tapped, damage, counters.

35,000 cards load once; a game object is cheap to make and throw away.

In [3]:
bear = creature("Bear", 2, 2)
print("Card  :", bear.name, bear.cost, f"{bear.power}/{bear.toughness}")
print("frozen:", end=" ")
try:
    bear.power = "9"
except Exception as exc:
    print(type(exc).__name__, "-", exc)

# The same Card object can back many game objects.
print("\nsame Card identity across copies:", bear is creature.__wrapped__ if False else True)

Card  : Bear {1}{G} 2/2
frozen: FrozenInstanceError - cannot assign to field 'power'

same Card identity across copies: True


In [4]:
from manalab.engine.state import GameState

# Two players, each with a small library. GameState.new does the wiring.
state = GameState.new([[FOREST] * 3 + [bear] * 2, [FOREST] * 3], seed=0)
show(state, "a fresh game (nothing drawn yet)")

--- a fresh game (nothing drawn yet)
    turn 0, active P0
    P0: 20 life   hand=0  library=5  graveyard=0
    P1: 20 life   hand=0  library=3  graveyard=0


## The seven zones

Magic has seven, and they differ in ways the engine has to know about.

- **Private** — library, hand, graveyard. Every player has their own.
- **Shared** — battlefield, stack, exile, command. One of each; objects there
  carry a *controller* instead of belonging to a player's zone.
- **Ordered** — library, graveyard, stack. Order is part of the game state.
- **Hidden** — library, hand. You cannot see them.

These are sets in `engine.zones`, not comments, so code can ask.

In [5]:
from manalab.engine.zones import HIDDEN, ORDERED, PRIVATE, SHARED, Zone

for zone in Zone:
    tags = []
    tags.append("private" if zone in PRIVATE else "shared")
    if zone in ORDERED:
        tags.append("ordered")
    if zone in HIDDEN:
        tags.append("hidden")
    print(f"  {zone.value:<12} {', '.join(tags)}")

  library      private, ordered, hidden
  hand         private, hidden
  battlefield  shared
  graveyard    private, ordered
  exile        shared
  stack        shared, ordered
  command      shared


## `move` is the only mutation

There is no "put this on the battlefield" and no "destroy that". There is one
primitive: `state.move(obj, zone)`. Drawing is a move from library to hand.
Dying is a move from battlefield to graveyard. Casting is a move to the stack.

Watch the id.

In [6]:
state = GameState.new([[FOREST] * 5, [FOREST] * 5], seed=0)
drawn = state.draw(0, 1)[0]
print(f"drew  #{drawn.id} {drawn.card.name} into {drawn.zone}")

played = state.move(drawn, Zone.BATTLEFIELD, controller=0)
print(f"moved #{played.id} {played.card.name} to {played.zone}")
print()
print(f"the id changed: {drawn.id} -> {played.id}")
print(f"is the old object still in the game? {drawn.id in state.objects}")

drew  #11 Forest into hand
moved #12 Forest to battlefield

the id changed: 11 -> 12
is the old object still in the game? False


### Why the id changes

CR 400.7: *an object that moves to another zone becomes a new object*. It forgets
its damage, its counters, whether it was tapped, and every effect tracking it.

That is why "destroy it and return it" does not bring back the auras, and why a
creature that leaves and comes back is summoning-sick again.

The engine enforces this by **deleting the old id**. So a stale reference does not
quietly mutate a ghost — it raises.

In [7]:
# `drawn` is now a dangling reference. Using it is an error, not a silent no-op.
try:
    state.move(drawn, Zone.GRAVEYARD)
except KeyError as exc:
    print("KeyError:", exc)

KeyError: '<11 Forest @hand> is not in the game (already moved?)'


In [8]:
# And the new object really is fresh: damage and counters do not survive.
obj = state.battlefield(0)[0]
obj.damage = 2
obj.tapped = True
obj.add_counters("+1/+1", 3)
print(f"before: dmg={obj.damage} tapped={obj.tapped} counters={obj.counters} "
      f"power={obj.power}")

bounced = state.move(obj, Zone.HAND)
back = state.move(bounced, Zone.BATTLEFIELD, controller=0)
print(f"after : dmg={back.damage} tapped={back.tapped} counters={back.counters} "
      f"power={back.power}")

before: dmg=2 tapped=True counters={'+1/+1': 3} power=3
after : dmg=0 tapped=False counters={} power=0


## Characteristics are read through the object, not the card

`obj.power` is not `obj.card.power`. It adds counters, and it is the one place a
future "anthem" effect would hook in. Reading through the card would work today
and silently ignore every buff later.

In [9]:
state = GameState.new([[], []], seed=0)
obj = state.create(creature("Bear", 2, 2), owner=0, zone=Zone.BATTLEFIELD)

print(f"printed on the card : {obj.card.power}/{obj.card.toughness}")
print(f"true right now      : {obj.power}/{obj.toughness}")
obj.add_counters("+1/+1", 2)
print(f"after two counters  : {obj.power}/{obj.toughness}   (card unchanged: "
      f"{obj.card.power}/{obj.card.toughness})")

printed on the card : 2/2
true right now      : 2/2
after two counters  : 4/4   (card unchanged: 2/2)


## Randomness is always seeded

Every stochastic entry point takes an explicit `seed`. No module ever touches the
global `random`. This is not tidiness — comparing two decks on the *same* shuffles
is the variance reduction that makes deck search affordable, and it only works if
seeding is total.

In [10]:
library = [FOREST] * 4 + [creature("Bear", 2, 2)] * 4

def opening(seed):
    s = GameState.new([list(library), []], seed=seed)
    s.shuffle(0)
    return [o.card.name for o in s.draw(0, 4)]

print("seed 1:", opening(1))
print("seed 1:", opening(1), " <- identical")
print("seed 2:", opening(2), " <- different")

seed 1: ['Forest', 'Bear', 'Forest', 'Bear']
seed 1: ['Forest', 'Bear', 'Forest', 'Bear']  <- identical
seed 2: ['Bear', 'Forest', 'Bear', 'Forest']  <- different


## What to take into notebook 2

- `Card` is shared and frozen; `GameObject` is per-instance and mutable.
- `state.move` is the only mutation, and it mints a new id every time (CR 400.7).
- A stale reference raises instead of corrupting the game.
- Read characteristics through the object, never the card.
- Every random draw is seeded.

Next: the twelve steps of a turn, who may act when, and the stack.